In [4]:
!pip install psycopg2-binary

In [5]:
!pip install kafka-python

In [3]:
import psycopg2
import logging
from kafka import KafkaConsumer
import json

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(message)s")
logger = logging.getLogger(__name__)

# ==============================
# KAFKA CONSUMER — fresh setiap run
# ==============================
consumer = KafkaConsumer(
    "orders",
    bootstrap_servers="kafka:29092",
    value_deserializer=lambda x: json.loads(x.decode("utf-8")),
    auto_offset_reset="earliest",
    group_id=None,               # ← None agar selalu baca dari awal
    consumer_timeout_ms=10000
)

# ==============================
# KONEKSI POSTGRESQL
# ==============================
conn = psycopg2.connect(
    host="postgres",
    database="olist_ecommerce",
    user="airflow",
    password="airflow",
    port="5432",
    options="-c search_path=raw"
)
cur = conn.cursor()
logger.info("🚀 Consumer jalan...")

# ==============================
# CONSUMER LOOP
# ==============================
total = 0
error_count = 0
BATCH_SIZE = 100
MAX_ERROR = 10

try:
    for message in consumer:
        data = message.value

        try:
            cur.execute("""
                INSERT INTO raw.raw_orders (
                    order_id, customer_id, order_status,
                    order_purchase_timestamp, order_approved_at,
                    order_delivered_carrier_date, order_delivered_customer_date,
                    order_estimated_delivery_date
                )
                VALUES (%s,%s,%s,%s,%s,%s,%s,%s)
                ON CONFLICT (order_id) DO NOTHING;
            """, (
                data.get("order_id"),
                data.get("customer_id"),
                data.get("order_status"),
                data.get("order_purchase_timestamp"),
                data.get("order_approved_at"),
                data.get("order_delivered_carrier_date"),
                data.get("order_delivered_customer_date"),
                data.get("order_estimated_delivery_date")
            ))
            total += 1

            if total % BATCH_SIZE == 0:
                conn.commit()
                logger.info(f"✅ {total} rows inserted...")

        except Exception as e:
            conn.rollback()
            error_count += 1
            logger.error(f"❌ Error row [{data.get('order_id')}]: {e}")

            if error_count >= MAX_ERROR:
                logger.critical("🚨 Too many errors! Stopping pipeline...")
                raise

except Exception as e:
    logger.critical(f"💀 Pipeline stopped: {e}")
    raise

finally:
    conn.commit()
    cur.close()
    conn.close()
    logger.info(f"🏁 Done. Total {total} rows inserted | {error_count} errors")

2026-03-28 14:06:59,044 - <BrokerConnection client_id=kafka-python-2.3.0, node_id=bootstrap-0 host=kafka:29092 <connecting> [IPv4 ('172.19.0.7', 29092)]>: connecting to kafka:29092 [('172.19.0.7', 29092) IPv4]
2026-03-28 14:06:59,048 - <BrokerConnection client_id=kafka-python-2.3.0, node_id=bootstrap-0 host=kafka:29092 <checking_api_versions_recv> [IPv4 ('172.19.0.7', 29092)]>: Broker version identified as 2.6
2026-03-28 14:06:59,049 - <BrokerConnection client_id=kafka-python-2.3.0, node_id=bootstrap-0 host=kafka:29092 <connected> [IPv4 ('172.19.0.7', 29092)]>: Connection complete.
2026-03-28 14:06:59,050 - group_id is None: disabling auto-commit.
2026-03-28 14:06:59,050 - Updating subscribed topics to: ('orders',)
2026-03-28 14:06:59,065 - 🚀 Consumer jalan...
2026-03-28 14:06:59,067 - Updated partition assignment: [TopicPartition(topic='orders', partition=0)]
2026-03-28 14:06:59,068 - <BrokerConnection client_id=kafka-python-2.3.0, node_id=1 host=kafka:29092 <connecting> [IPv4 ('172.1